
# Caso de clase CAPM — Parte 2
## ¿Beta explica realmente los retornos?

Esta segunda parte profundiza en el CAPM utilizando:

- acciones individuales;
- ETF;
- fondos de inversión;
- acciones con niveles de precio y volatilidad muy diferentes.

El objetivo es separar cuidadosamente:

$$
\text{precio}
$$

$$
\text{volatilidad total}
$$

$$
\text{beta}
$$

$$
\text{retorno esperado}
$$

y estudiar si el CAPM funciona de la misma forma para distintos tipos de activos.


In [ ]:
# Si hace falta:
# !pip install yfinance pandas numpy matplotlib statsmodels

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf
import statsmodels.api as sm

pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

## 1. Datos del caso

In [ ]:
start_date = "2018-01-01"
end_date   = "2025-12-31"

stocks = {
    "Tesla": "TSLA",
    "Nvidia": "NVDA",
    "Berkshire_B": "BRK-B",
    "Booking": "BKNG",
    "Ford": "F",
    "Snap": "SNAP",
    "CocaCola": "KO",
    "JPMorgan": "JPM"
}

etfs = {
    "SP500": "SPY",
    "Nasdaq100": "QQQ",
    "SmallCap": "IWM",
    "Technology": "XLK",
    "Healthcare": "XLV",
    "Energy": "XLE"
}

funds = {
    "Vanguard500Fund": "VFIAX",
    "VanguardTotalMarket": "VTSAX",
    "FidelityContrafund": "FCNTX"
}

all_assets = {**stocks, **etfs, **funds}
ticker_to_name = {v: k for k, v in all_assets.items()}

In [ ]:
def download_close(ticker, start, end):
    data = yf.download(
        ticker,
        start=start,
        end=end,
        auto_adjust=True,
        progress=False
    )

    if data.empty:
        return None

    close = data["Close"]

    if isinstance(close, pd.DataFrame):
        close = close.iloc[:, 0]

    close.name = ticker
    return close

series = []

for ticker in all_assets.values():
    s = download_close(ticker, start_date, end_date)
    if s is not None:
        series.append(s)

prices = pd.concat(series, axis=1).rename(columns=ticker_to_name)

monthly_prices = prices.resample("ME").last()
returns = monthly_prices.pct_change(fill_method=None)

print("Activos disponibles:")
print(list(prices.columns))

## 2. Tasa libre de riesgo y CAPM

In [ ]:
rf_annual = 0.04
rf_monthly = (1 + rf_annual)**(1/12) - 1

market = "SP500"
excess_returns = returns - rf_monthly


La ecuación empírica es:

$$
R_{i,t}-R_{f,t}
=
\alpha_i
+
\beta_i(R_{m,t}-R_{f,t})
+
\varepsilon_{i,t}
$$

y la predicción de equilibrio es:

$$
E(R_i)
=
R_f
+
\beta_i
\left[
E(R_m)-R_f
\right]
$$


In [ ]:
def estimate_capm(asset, excess_returns, market="SP500"):

    data = pd.concat(
        [
            excess_returns[asset].rename("asset"),
            excess_returns[market].rename("market")
        ],
        axis=1
    ).dropna()

    y = data["asset"]
    X = sm.add_constant(data["market"])

    model = sm.OLS(y, X).fit()

    return {
        "Asset": asset,
        "N": len(data),
        "Alpha_monthly": model.params["const"],
        "Beta": model.params["market"],
        "Alpha_t": model.tvalues["const"],
        "Beta_t": model.tvalues["market"],
        "Alpha_pvalue": model.pvalues["const"],
        "Beta_pvalue": model.pvalues["market"],
        "R2": model.rsquared
    }, model

## 3. Tabla CAPM completa por tipo de activo

In [ ]:
rows = []
models = {}

for asset in stocks.keys():
    if asset in excess_returns.columns:
        r, m = estimate_capm(asset, excess_returns, market)
        r["Type"] = "Stock"
        rows.append(r)
        models[asset] = m

for asset in etfs.keys():
    if asset in excess_returns.columns and asset != market:
        r, m = estimate_capm(asset, excess_returns, market)
        r["Type"] = "ETF"
        rows.append(r)
        models[asset] = m

for asset in funds.keys():
    if asset in excess_returns.columns:
        r, m = estimate_capm(asset, excess_returns, market)
        r["Type"] = "Fund"
        rows.append(r)
        models[asset] = m

capm_results = pd.DataFrame(rows).set_index("Asset")
capm_results.sort_values(["Type", "Beta"], ascending=[True, False])

## 4. Agregar precio y volatilidad

In [ ]:
descriptive = pd.DataFrame({
    "Latest_Price": prices.ffill().iloc[-1],
    "Annualized_Return": returns.mean() * 12,
    "Annualized_Volatility": returns.std() * np.sqrt(12)
})

capm_full = capm_results.join(descriptive)

capm_full[
    [
        "Type",
        "Latest_Price",
        "Annualized_Volatility",
        "Beta",
        "R2",
        "Alpha_monthly",
        "Alpha_pvalue"
    ]
].sort_values("Beta", ascending=False)

## 5. ¿Precio alto significa beta alto?

In [ ]:
stocks_only = capm_full[capm_full["Type"] == "Stock"].dropna(
    subset=["Latest_Price", "Beta"]
)

fig, ax = plt.subplots(figsize=(9, 6))

ax.scatter(
    stocks_only["Latest_Price"],
    stocks_only["Beta"]
)

for asset in stocks_only.index:
    ax.annotate(
        asset,
        (
            stocks_only.loc[asset, "Latest_Price"],
            stocks_only.loc[asset, "Beta"]
        ),
        xytext=(5, 5),
        textcoords="offset points"
    )

ax.axhline(1, linewidth=1)
ax.set_xlabel("Precio nominal")
ax.set_ylabel("Beta")
ax.set_title("Precio nominal vs. beta")

plt.tight_layout()
plt.show()


La relación relevante del CAPM no contiene el precio nominal:

$$
\beta_i
=
\frac{\operatorname{Cov}(R_i,R_m)}
{\operatorname{Var}(R_m)}
$$

Por tanto, comparar Booking o Berkshire con Ford o Snap permite mostrar que el nivel del precio no determina el riesgo sistemático.


## 6. ¿Alta volatilidad implica beta alto?

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))

ax.scatter(
    stocks_only["Annualized_Volatility"],
    stocks_only["Beta"]
)

for asset in stocks_only.index:
    ax.annotate(
        asset,
        (
            stocks_only.loc[asset, "Annualized_Volatility"],
            stocks_only.loc[asset, "Beta"]
        ),
        xytext=(5, 5),
        textcoords="offset points"
    )

ax.axhline(1, linewidth=1)
ax.set_xlabel("Volatilidad anualizada")
ax.set_ylabel("Beta")
ax.set_title("Volatilidad total vs. beta")

plt.tight_layout()
plt.show()


La volatilidad total puede pensarse, bajo la regresión de mercado, como la combinación de un componente sistemático y uno idiosincrático:

$$
R_i-R_f
=
\alpha_i
+
\beta_i(R_m-R_f)
+
\varepsilon_i
$$

Por eso dos activos con volatilidades similares pueden tener betas muy distintos.


## 7. Acciones vs. ETF vs. fondos

In [ ]:
type_summary = capm_full.groupby("Type").agg(
    Mean_Beta=("Beta", "mean"),
    Median_Beta=("Beta", "median"),
    Mean_R2=("R2", "mean"),
    Mean_Volatility=("Annualized_Volatility", "mean"),
    Mean_Alpha=("Alpha_monthly", "mean")
)

type_summary


Esta tabla permite discutir una diferencia importante:

- una acción individual puede contener una gran cantidad de riesgo idiosincrático;
- un ETF o fondo diversificado suele reflejar con mayor claridad exposiciones sistemáticas;
- un $R^2$ más alto significa que el mercado explica una mayor proporción de sus movimientos temporales.


## 8. Gráfica de beta por tipo de activo

In [ ]:
plot_data = capm_full.sort_values("Beta")

fig, ax = plt.subplots(figsize=(10, 7))

ax.barh(plot_data.index, plot_data["Beta"])
ax.axvline(1, linewidth=1)

ax.set_xlabel("Beta")
ax.set_ylabel("Activo")
ax.set_title("Comparación de beta: acciones, ETF y fondos")

plt.tight_layout()
plt.show()

## 9. Security Market Line


El CAPM predice:

$$
E(R_i)
=
R_f
+
\beta_i
\left[
E(R_m)-R_f
\right]
$$

Esto genera la **Security Market Line (SML)**.


In [ ]:
market_return_annual = returns[market].mean() * 12

capm_full["Observed_Return"] = capm_full["Annualized_Return"]

capm_full["CAPM_Return"] = (
    rf_annual
    + capm_full["Beta"] * (market_return_annual - rf_annual)
)

capm_full["Pricing_Error"] = (
    capm_full["Observed_Return"]
    - capm_full["CAPM_Return"]
)

capm_full[
    ["Type", "Beta", "Observed_Return", "CAPM_Return", "Pricing_Error"]
].sort_values("Beta")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 7))

beta_min = min(0, capm_full["Beta"].min() - 0.1)
beta_max = capm_full["Beta"].max() + 0.2

beta_grid = np.linspace(beta_min, beta_max, 100)

sml = (
    rf_annual
    + beta_grid * (market_return_annual - rf_annual)
)

ax.plot(beta_grid, sml, label="Security Market Line")

ax.scatter(
    capm_full["Beta"],
    capm_full["Observed_Return"]
)

for asset in capm_full.index:
    ax.annotate(
        asset,
        (
            capm_full.loc[asset, "Beta"],
            capm_full.loc[asset, "Observed_Return"]
        ),
        xytext=(4, 4),
        textcoords="offset points"
    )

ax.set_xlabel("Beta")
ax.set_ylabel("Retorno promedio anualizado")
ax.set_title("Retorno observado vs. retorno predicho por CAPM")
ax.legend()

plt.tight_layout()
plt.show()


Si un activo está por encima de la SML:

$$
R_i^{obs}
>
R_i^{CAPM}
$$

mientras que si está por debajo:

$$
R_i^{obs}
<
R_i^{CAPM}
$$

Esto no prueba automáticamente que el activo esté mal valorado. También puede reflejar:

- error muestral;
- otros factores de riesgo;
- inestabilidad de beta;
- una mala proxy del mercado;
- limitaciones del CAPM.


## 10. Alpha de Jensen

In [ ]:
capm_full["Jensen_Alpha_Annual"] = (
    (1 + capm_full["Alpha_monthly"])**12 - 1
)

capm_full[
    [
        "Type",
        "Beta",
        "Jensen_Alpha_Annual",
        "Alpha_t",
        "Alpha_pvalue"
    ]
].sort_values("Jensen_Alpha_Annual", ascending=False)


El alpha de Jensen representa:

$$
\alpha_i
=
R_i
-
\left[
R_f
+
\beta_i(R_m-R_f)
\right]
$$

Para fondos y ETF esta medida es particularmente útil como benchmark de desempeño ajustado por riesgo sistemático.


## 11. Sharpe y Treynor para acciones, ETF y fondos

In [ ]:
performance = pd.DataFrame({
    "Annual_Return": returns.mean() * 12,
    "Annual_Volatility": returns.std() * np.sqrt(12)
})

performance["Sharpe"] = (
    performance["Annual_Return"] - rf_annual
) / performance["Annual_Volatility"]

for asset in capm_full.index:
    performance.loc[asset, "Beta"] = capm_full.loc[asset, "Beta"]

performance.loc[market, "Beta"] = 1.0

performance["Treynor"] = (
    performance["Annual_Return"] - rf_annual
) / performance["Beta"]

performance.loc[
    [a for a in performance.index if a in capm_full.index],
    ["Annual_Return", "Annual_Volatility", "Beta", "Sharpe", "Treynor"]
].sort_values("Sharpe", ascending=False)


Sharpe utiliza riesgo total:

$$
SR_i
=
\frac{R_i-R_f}{\sigma_i}
$$

Treynor utiliza riesgo sistemático:

$$
TR_i
=
\frac{R_i-R_f}{\beta_i}
$$

La comparación es especialmente interesante entre una acción individual y un fondo diversificado.


## 12. Estabilidad temporal del beta

In [ ]:
def rolling_beta(asset, market, excess_returns, window=36):

    pair = pd.concat(
        [
            excess_returns[asset].rename("asset"),
            excess_returns[market].rename("market")
        ],
        axis=1
    )

    cov_rolling = pair["asset"].rolling(window).cov(pair["market"])
    var_market = pair["market"].rolling(window).var()

    return cov_rolling / var_market

selected_for_rolling = [
    a for a in [
        "Tesla",
        "Nvidia",
        "CocaCola",
        "Booking",
        "Nasdaq100",
        "SmallCap",
        "Vanguard500Fund"
    ]
    if a in excess_returns.columns
]

rolling_betas = pd.DataFrame(index=excess_returns.index)

for asset in selected_for_rolling:
    rolling_betas[asset] = rolling_beta(
        asset,
        market,
        excess_returns,
        window=36
    )

rolling_betas.tail()

In [ ]:
fig, ax = plt.subplots(figsize=(12, 7))

rolling_betas.plot(ax=ax)

ax.axhline(1, linewidth=1)
ax.set_title("Beta móvil de 36 meses")
ax.set_xlabel("Fecha")
ax.set_ylabel("Beta")

plt.tight_layout()
plt.show()


Si:

$$
\beta_{i,t}
$$

cambia a través del tiempo, usar un único beta histórico implica asumir una estabilidad que puede no existir.

Esto introduce una de las debilidades del CAPM estático.


## 13. Tabla pedagógica final

In [ ]:
final_table = capm_full[
    [
        "Type",
        "Latest_Price",
        "Annualized_Volatility",
        "Beta",
        "R2",
        "Observed_Return",
        "CAPM_Return",
        "Pricing_Error",
        "Jensen_Alpha_Annual",
        "Alpha_pvalue"
    ]
].copy()

final_table.sort_values(["Type", "Beta"], ascending=[True, False])


# 14. Preguntas para discusión en clase

1. ¿Booking tiene un beta alto simplemente porque su acción tiene un precio nominal alto?
2. ¿Ford tiene poco riesgo porque su precio por acción es bajo?
3. ¿Tesla y Nvidia tienen beta alto porque son volátiles o porque covarían fuertemente con el mercado?
4. ¿Por qué un ETF puede tener un $R^2$ mayor que una acción individual?
5. ¿Un fondo con $\alpha>0$ necesariamente genera habilidad del administrador?
6. ¿Qué ocurre si $\beta$ cambia sustancialmente según la ventana de estimación?
7. ¿El S&P 500 es realmente “el mercado” que exige la teoría CAPM?

La siguiente parte puede utilizar estas preguntas para introducir:

$$
\text{tests transversales}
\rightarrow
\text{Fama-MacBeth}
\rightarrow
\text{Fama-French}
\rightarrow
\text{crítica de Roll}
$$
